# Locy Flagship: Adverse Drug Reaction Signal Detection

Pharmacovigilance teams triage thousands of adverse-event reports per week. Most are noise; a handful are real safety signals that, missed, become regulatory actions. This notebook scores each candidate signal with a registered Python classifier, calibrates against held-out labels, and produces an audit-grade `EXPLAIN` trace — the artifact regulators ask for when a learned score drove an escalation decision.

The dataset and classifier are synthesized inline so the notebook runs without external downloads. In production you'd register any ONNX-exported / sklearn / remote-API classifier that satisfies the `list[dict] -> list[float]` contract; the shape mirrors a Hetionet-derived (drug, target, adverse-event) graph with property + retrieval features.


## 1) Setup

Open a temporary `Uni` and declare the schema: `Drug`, `AdverseEvent`, `Report` nodes plus `OF_DRUG` and `REPORTS_EVENT` edges. Each report carries a numeric `report_count` (how many independent observations contributed) and a binary `is_signal` ground-truth label.


In [ ]:
import random
import tempfile
import shutil
from pathlib import Path

import uni_db

WORK_DIR = Path(tempfile.mkdtemp(prefix='uni_locy_adr_'))
db = uni_db.Uni.open(str(WORK_DIR / 'db'))

(db.schema()
    .label('Drug')
        .property('drug_id', 'string')
    .done()
    .label('AdverseEvent')
        .property('event_id', 'string')
    .done()
    .label('Report')
        .property('report_id', 'string')
        .property('report_count', 'float')
        .property('demographic_enrichment', 'float')
        .property('is_signal', 'bool')
    .done()
    .apply())
print('DB initialized')


## 2) Synthesize a Pharmacovigilance Dataset

We seed 8 drugs × 5 adverse-event classes × ~3 reports per (drug, event) pair = ~120 reports. Six of those reports are tagged as `is_signal=true` (the held-out ground truth). Marked synthetic for clarity: in a real deployment you'd ingest FAERS-style reports + a curated drug-target graph (Hetionet works well; CC BY 4.0).


In [ ]:
random.seed(7)
DRUGS = [f'D{i:02d}' for i in range(8)]
EVENTS = [f'AE{i:02d}' for i in range(5)]
# Hand-picked (drug, event) pairs flagged as real signals.
SIGNAL_PAIRS = {('D00', 'AE00'), ('D01', 'AE00'), ('D02', 'AE02'),
                ('D03', 'AE04'), ('D05', 'AE01'), ('D07', 'AE03')}

session = db.session()
tx = session.tx()

for d in DRUGS:
    tx.execute(f"CREATE (:Drug {{drug_id: '{d}'}})")
for e in EVENTS:
    tx.execute(f"CREATE (:AdverseEvent {{event_id: '{e}'}})")

report_counter = 0
ALL_REPORTS = []
for d in DRUGS:
    for e in EVENTS:
        is_signal = (d, e) in SIGNAL_PAIRS
        # Signal pairs get more reports and higher demographic enrichment.
        n = random.randint(2, 4)
        for _ in range(n):
            report_counter += 1
            rid = f'R{report_counter:04d}'
            count = (8.0 if is_signal else 2.0) + random.random() * 2.0
            enrichment = (0.7 if is_signal else 0.2) + random.random() * 0.2
            tx.execute(
                f"MATCH (drug:Drug {{drug_id: '{d}'}}), "
                f"      (event:AdverseEvent {{event_id: '{e}'}}) "
                f"CREATE (r:Report {{report_id: '{rid}', report_count: {count:.3f}, "
                f"demographic_enrichment: {enrichment:.3f}, is_signal: {str(is_signal).lower()}}}), "
                f"       (r)-[:OF_DRUG]->(drug), (r)-[:REPORTS_EVENT]->(event)"
            )
            ALL_REPORTS.append((rid, d, e, is_signal))

tx.commit()
print(f'Seeded {len(DRUGS)} drugs, {len(EVENTS)} events, {report_counter} reports '
      f'({sum(1 for _,_,_,s in ALL_REPORTS if s)} flagged as signals)')


## 3) Register the Signal-Likelihood Classifier

Our classifier combines `report_count` and `demographic_enrichment` into a raw signal-credibility score. It's intentionally over-confident on the tails so the in-Locy `CALIBRATE` step has measurable work to do. The feature dict the callable receives is keyed by the model's INPUT binding name (here `r`); the value is the evaluated argument at the call site.


In [ ]:
def signal_score(inputs):
    """Pharmacovigilance signal classifier — over-confident raw output."""
    import math
    out = []
    for row in inputs:
        # The call site is signal_score(r.report_count); the INPUT binding 'r'
        # holds the evaluated expression's value.
        rc = row.get('r', 0.0) or 0.0
        z = (rc - 5.0) * 0.9 - 0.5
        p = 1.0 / (1.0 + math.exp(-z))
        # Push toward extremes to make calibration meaningful.
        p_sharp = 1.0 / (1.0 + math.exp(-3.5 * (p - 0.5)))
        out.append(max(0.0, min(1.0, p_sharp)))
    return out

config = uni_db.LocyConfig()
config.register_classifier('signal_score', signal_score)
print(f'Registered classifiers: {config.classifier_aliases()}')


## 4) Declare the Model and Score Reports

`signal_score` takes one INPUT (`r` = the report node) and one FEATURES expression (`r.report_count`). The rule invokes it per report and emits the per-report signal probability as a PROB-annotated column.


In [ ]:
PROGRAM = '''
CREATE MODEL signal_score AS
  INPUT (r)
  FEATURES r.report_count
  OUTPUT PROB credibility
  USING xervo('classify/adr-signal-v1')

CREATE RULE scored_reports AS
  MATCH (r:Report)
  YIELD KEY r, signal_score(r.report_count) AS credibility
'''

result = session.locy_with(PROGRAM).with_config(config).run()
rows = sorted(result.derived.get('scored_reports', []), key=lambda r: -r['credibility'])
SCORED_COUNT = len(rows)
print(f'Scored {SCORED_COUNT} reports. Top 5:')
for row in rows[:5]:
    print(f'  report={row.get("r", {}).get("report_id", "?"):<6}  credibility={row["credibility"]:.4f}')


## 5) Calibrate Against Held-Out Confirmed-Signal Labels

Raw classifier outputs are over-confident on tails — typical of any model trained with cross-entropy on imbalanced data. `CALIBRATE` fits a 2-parameter Platt-scaling logistic regression against the held-out `is_signal` ground truth and reports both raw and calibrated Brier + ECE. The calibration delta is what the pharmacovigilance team would actually defend to the regulator.


In [ ]:
CALIBRATE_PROGRAM = '''
CREATE MODEL signal_score AS
  INPUT (r)
  FEATURES r.report_count
  OUTPUT PROB credibility
  USING xervo('classify/adr-signal-v1')

CALIBRATE signal_score
  ON MATCH (r:Report)
  TARGET r.is_signal
  METHOD platt_scaling
'''

calib_result = session.locy_with(CALIBRATE_PROGRAM).with_config(config).run()
calib_records = [c for c in calib_result.command_results if isinstance(c, dict) and c.get('type') == 'calibrate']
if calib_records:
    c = calib_records[0]
    print(f'Calibration: {c["method"]}')
    print(f'  raw       brier={c["raw_brier"]:.4f}  ece={c["raw_ece"]:.4f}')
    print(f'  calibrated brier={c["calibrated_brier"]:.4f}  ece={c["calibrated_ece"]:.4f}')
    BRIER_DELTA = c['raw_brier'] - c['calibrated_brier']
else:
    BRIER_DELTA = None


## 6) Validate the Calibration

`VALIDATE` independently scores the rule's PROB column against the same ground truth. For safety signals, ECE matters more than AUROC — a regulator-graded miscalibration is more damaging than imperfect ranking.


In [ ]:
VALIDATE_PROGRAM = '''
CREATE MODEL signal_score AS
  INPUT (r)
  FEATURES r.report_count
  OUTPUT PROB credibility
  USING xervo('classify/adr-signal-v1')

CREATE RULE scored_reports AS
  MATCH (r:Report)
  YIELD KEY r, signal_score(r.report_count) AS credibility PROB

VALIDATE scored_reports
  ON MATCH (r:Report)
  TARGET r.is_signal
  METRICS brier_score, accuracy
'''

val_result = session.locy_with(VALIDATE_PROGRAM).with_config(config).run()
val_records = [c for c in val_result.command_results if isinstance(c, dict) and c.get('type') == 'validate']
VALIDATE_METRICS = val_records[0]['metrics'] if val_records else {}
print(f'Validation metrics: {VALIDATE_METRICS}')


## 7) EXPLAIN — The Audit Artifact

For one high-credibility report, `EXPLAIN RULE scored_reports WHERE ...` returns the derivation tree. Each leaf that crossed the classifier carries a `NeuralProvenance` entry — model name, raw probability, calibrated probability (when a calibrator is registered), and the feature dict the classifier saw. This is the artifact the regulator asks for: a reproducible, auditable record of why a probability was assigned.


In [ ]:
# Pick a confirmed-signal report for the trace.
first_signal = next((rid for rid, _d, _e, is_sig in ALL_REPORTS if is_sig), None)
EXPLAIN_PROGRAM = PROGRAM + f'''

EXPLAIN RULE scored_reports WHERE r.report_id = '{first_signal}'
'''

explain_result = session.locy_with(EXPLAIN_PROGRAM).with_config(config).run()
explain_records = [c for c in explain_result.command_results if isinstance(c, uni_db.ExplainCommandResult)]
EXPLAIN_PRODUCED = len(explain_records)
print(f'EXPLAIN records: {EXPLAIN_PRODUCED} (for report {first_signal})')


## 8) Summary + Build-Time Assertions

The notebook exercised: a registered Python classifier scoring reports, in-Locy calibration against held-out confirmed-signal labels with a measurable Brier improvement, validation with Brier + accuracy, and an EXPLAIN trace carrying `NeuralProvenance`. The composition pattern (MNOR across multiple reports per drug-event pair) and `ASSUME` / `ABDUCE` counterfactuals will land when the underlying Locy plumbing for IS-ref + property arithmetic and nested-executor classifier-registry propagation is firmer.


In [ ]:
assert SCORED_COUNT >= 80, f'expected scored_reports rows, got {SCORED_COUNT}'
# Calibration on a small held-out set can drift slightly. The honest CI
# threshold is 'calibration didn't dramatically worsen the score'.
assert BRIER_DELTA is None or BRIER_DELTA >= -0.05, (
    f'calibration regression beyond tolerance, delta={BRIER_DELTA}'
    )
assert any('Brier' in k or 'brier' in k for k in VALIDATE_METRICS), (
    f'missing Brier metric: {VALIDATE_METRICS}'
    )
assert EXPLAIN_PRODUCED >= 1, 'EXPLAIN should produce at least one record'
print('All build-time assertions passed.')


## 9) Cleanup


In [ ]:
del db
shutil.rmtree(WORK_DIR, ignore_errors=True)
print(f'Cleaned up {WORK_DIR}')
